# NB05 — ESM-2 Benchmark & Active Learning

**Objective:** Benchmark Meta's ESM-2 protein language model embeddings against hand-crafted aa+kmer2 features, and apply uncertainty sampling to identify which ENIGMA organisms should be prioritized for targeted profiling.

## Background

**ESM-2** (Evolutionary Scale Modeling) is Meta's 650M-parameter protein language model pre-trained on 2.7B protein sequences. It produces dense 1280-dimensional embeddings capturing evolutionary and functional information.

**Active Learning** via uncertainty sampling identifies training examples near the model's decision boundary, prioritizing which new organisms to profile. This guides experimental resource allocation.

## Workflow

1. Load or scaffold ESM-2 embeddings (1280-dim) from protein sequences
2. Benchmark ESM-2 on binary essentiality classification (vs. baseline 0.659 AUC)
3. Combine ESM-2 + aa/kmer2 features; evaluate joint performance
4. Apply uncertainty sampling on training data using existing CatBoost model
5. Rank organisms by: (mean uncertainty × n_proteins) / LOGO_AUC
6. Recommend top-5 organisms for targeted profiling with rationale

## Inputs

- `data/X_train.npy`, `data/X_test.npy` — baseline 420-dim features
- `data/y_train.npy`, `data/y_test.npy` — binary essentiality labels
- `data/general_essentiality_model.cbm` — trained CatBoost (test AUC = 0.659, LOGO AUC = 0.618±0.037)
- `data/logo_results.csv` — genus-level LOGO cross-validation results
- `data/labeled_train.parquet` — organism and genus metadata
- `data/esm2_embeddings_train.npy` (optional) — pre-extracted ESM-2 embeddings (166705×1280)

## Outputs

- `data/active_learning_priorities.csv` — ranked organisms by uncertainty and performance
- Comparison table: baseline vs ESM-2 vs combined AUC
- Top-5 recommendations for experimental targeting

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings('ignore')

from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score

PROJ_ROOT = Path.cwd().parent
DATA_DIR  = PROJ_ROOT / 'data'
FIG_DIR   = PROJ_ROOT / 'figures'
FIG_DIR.mkdir(exist_ok=True)

print(f"Project root: {PROJ_ROOT}")
print(f"Data directory: {DATA_DIR}")

## 1. ESM-2 embedding extraction

In [ ]:
esm2_train_file = DATA_DIR / 'esm2_embeddings_train.npy'
esm2_test_file = DATA_DIR / 'esm2_embeddings_test.npy'

esm2_available = esm2_train_file.exists() and esm2_test_file.exists()

if esm2_available:
    print(f"Loading ESM-2 embeddings...")
    embeddings_train = np.load(esm2_train_file)
    embeddings_test = np.load(esm2_test_file)
    print(f"ESM-2 train shape: {embeddings_train.shape}")
    print(f"ESM-2 test shape:  {embeddings_test.shape}")
else:
    print("ESM-2 embeddings not found.")
    print("\n=== ESM-2 Extraction Guide ===")
    print("\nESM-2 requires GPU and the fair-esm package:")
    print("  pip install fair-esm torch")
    print("\nModel: esm2_t33_650M_UR50D (650M params, 1280-dim embeddings)")
    print("\nExtraction script (pseudo-code):")
    print("  ")
    print("  import torch")
    print("  from esm.pretrained import esmfold_v1")
    print("  model, alphabet = torch.hub.load('facebookresearch/esm:main', 'esm2_t33_650M_UR50D')")
    print("  # Batch process sequences from FitnessBrowser")
    print("  # For each protein sequence: embeddings = model(sequence, repr_layers=[33])")
    print("  # Save to data/esm2_embeddings_train.npy (166705 × 1280)")
    print("  # Save to data/esm2_embeddings_test.npy (48346 × 1280)")
    print("\n")
    print("To proceed with analysis:")
    print("  1. Run extraction on GPU cluster")
    print("  2. Save embeddings as .npy arrays")
    print("  3. Rerun this cell")

## 2. ESM-2 feature benchmark

In [ ]:
# Load baseline features and labels
X_train = np.load(DATA_DIR / 'X_train.npy')
X_test = np.load(DATA_DIR / 'X_test.npy')
y_train = np.load(DATA_DIR / 'y_train.npy')
y_test = np.load(DATA_DIR / 'y_test.npy')

# Load baseline model
model_baseline = CatBoostClassifier()
model_baseline.load_model(str(DATA_DIR / 'general_essentiality_model.cbm'))

y_pred_baseline = model_baseline.predict_proba(X_test)[:, 1]
auc_baseline = roc_auc_score(y_test, y_pred_baseline)

print(f"=== Baseline Model (aa+kmer2) ===")
print(f"Test AUC: {auc_baseline:.4f}")
print(f"Features: 420 (20 AA + 400 kmer2)")

# ESM-2 benchmark (if available)
if esm2_available:
    print(f"\n=== Training on ESM-2 embeddings ===")
    
    n_pos = y_train.sum()
    n_neg = (y_train == 0).sum()
    scale_pos_weight = n_neg / n_pos
    
    model_esm2 = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        scale_pos_weight=scale_pos_weight,
        eval_metric='AUC',
        random_seed=42,
        verbose=100,
    )
    model_esm2.fit(embeddings_train, y_train)
    
    y_pred_esm2 = model_esm2.predict_proba(embeddings_test)[:, 1]
    auc_esm2 = roc_auc_score(y_test, y_pred_esm2)
    
    print(f"Test AUC: {auc_esm2:.4f}")
    print(f"Features: 1280 (ESM-2 embeddings)")
    print(f"\nImprovement vs baseline: {auc_esm2 - auc_baseline:+.4f} AUC")
else:
    print(f"\nESM-2 embeddings unavailable. Run extraction first (Section 1).")
    auc_esm2 = None

## 3. ESM-2 + aa/kmer2 combined features

In [ ]:
if esm2_available:
    print(f"=== Combined Features (ESM-2 + aa/kmer2) ===")
    
    # Concatenate embeddings with baseline features
    X_train_combined = np.hstack([embeddings_train, X_train])
    X_test_combined = np.hstack([embeddings_test, X_test])
    
    print(f"Combined feature shape: {X_train_combined.shape}")
    print(f"  1280 (ESM-2) + 420 (aa+kmer2) = 1700 dimensions")
    
    model_combined = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        scale_pos_weight=scale_pos_weight,
        eval_metric='AUC',
        random_seed=42,
        verbose=100,
    )
    model_combined.fit(X_train_combined, y_train)
    
    y_pred_combined = model_combined.predict_proba(X_test_combined)[:, 1]
    auc_combined = roc_auc_score(y_test, y_pred_combined)
    
    print(f"\nTest AUC: {auc_combined:.4f}")
    print(f"Improvement vs baseline: {auc_combined - auc_baseline:+.4f} AUC")
    print(f"\nComparison summary:")
    print(f"  Baseline (aa+kmer2):   {auc_baseline:.4f}")
    print(f"  ESM-2 only:            {auc_esm2:.4f}")
    print(f"  Combined (1700-dim):   {auc_combined:.4f}")
else:
    print("ESM-2 embeddings unavailable. Skipping combined feature analysis.")

## 4. Active learning: uncertainty sampling

In [ ]:
# Load metadata
df_train = pd.read_parquet(DATA_DIR / 'labeled_train.parquet')
df_logo = pd.read_csv(DATA_DIR / 'logo_results.csv')

# Get predicted probabilities on training set
y_pred_proba_train = model_baseline.predict_proba(X_train)[:, 1]

# Compute uncertainty = distance from decision boundary (max at p=0.5)
uncertainty = 0.5 - np.abs(y_pred_proba_train - 0.5)

# Add to metadata
df_train['uncertainty'] = uncertainty
df_train['pred_proba'] = y_pred_proba_train

# Group by organism and compute statistics
org_stats = df_train.groupby('organism').agg({
    'uncertainty': 'mean',
    'protein_id': 'count',  # n_proteins
    'essential_union': ['sum', 'mean'],
}).reset_index()

org_stats.columns = ['organism', 'mean_uncertainty', 'n_proteins', 'n_essential', 'essential_rate']
org_stats['genus'] = df_train.groupby('organism')['genus'].first().values

print(f"Organism-level uncertainty statistics:")
print(org_stats.head(10))

## 5. Proposed experimental targets

In [ ]:
# Join with LOGO results to get genus-level AUC
org_stats = org_stats.merge(
    df_logo[['genus', 'auc', 'low_n_flag']],
    on='genus',
    how='left'
)

# Fill NaN LOGO AUC with median for low-n genera
logo_auc_median = df_logo[~df_logo['low_n_flag']]['auc'].median()
org_stats['auc'] = org_stats['auc'].fillna(logo_auc_median)

# Compute priority score: (mean_uncertainty × n_proteins) / LOGO_AUC
# Higher = more uncertain, larger population, worse genus-level performance
org_stats['priority_score'] = (org_stats['mean_uncertainty'] * org_stats['n_proteins']) / org_stats['auc']

# Sort by priority
org_stats_ranked = org_stats.sort_values('priority_score', ascending=False)

# Top-5 recommendations
top5 = org_stats_ranked.head(5).copy()

print("=== TOP-5 ACTIVE LEARNING RECOMMENDATIONS ===")
print()
for rank, (idx, row) in enumerate(top5.iterrows(), 1):
    print(f"{rank}. {row['organism']} ({row['genus']})")
    print(f"   Mean uncertainty: {row['mean_uncertainty']:.4f}")
    print(f"   Genus LOGO AUC: {row['auc']:.4f}")
    print(f"   N proteins in training: {row['n_proteins']:.0f}")
    print(f"   Current essential rate: {100*row['essential_rate']:.2f}%")
    print(f"   Priority score: {row['priority_score']:.2f}")
    print()

# Save full ranking
org_stats_ranked.to_csv(DATA_DIR / 'active_learning_priorities.csv', index=False)
print(f"Saved full ranking to data/active_learning_priorities.csv")

## 6. Summary

In [ ]:
print("=== ANALYSIS SUMMARY ===")
print()
print("## Feature Benchmark Results")
print(f"Baseline (aa+kmer2):    {auc_baseline:.4f}")
if esm2_available:
    print(f"ESM-2 only:             {auc_esm2:.4f} (Δ {auc_esm2 - auc_baseline:+.4f})")
    print(f"Combined (1700-dim):    {auc_combined:.4f} (Δ {auc_combined - auc_baseline:+.4f})")
else:
    print("ESM-2 and combined results pending embedding extraction.")

print()
print("## Active Learning Recommendations")
print(f"\nTop-5 organisms for targeted profiling:")
for idx, (i, row) in enumerate(top5.iterrows(), 1):
    print(f"  {idx}. {row['organism']} (priority={row['priority_score']:.2f})")

print()
print("## Interpretation")
print("")
print("Active learning identifies organisms where:")
print("  • The model is uncertain (probabilities near 0.5)")
print("  • The population is large (more training examples)")
print("  • The genus-level model is weak (low LOGO AUC)")
print()
print("Profiling these organisms would provide:")
print("  1. Informative training examples for the model")
print("  2. Improved performance for their genus")
print("  3. Better generalization across organisms")